# 05. 서비스용 최종 모델: 로지스틱 회귀 (설정 고정)
01~04에서 정한 것을 그대로 고정해서 쓴다. **모델 비교·하이퍼파라미터 탐색 없음**
- 입력 변수 11개 (제안서 3.6절)
- 예측기간별 C: 1주 0.01 / 2주 0.3 / 4주 0.01 (01에서 valid PR-AUC로 고른 값)
- 결측: 중앙값 대체, 표준화, class_weight='balanced'
- 학습 데이터: 01과 같은 train(시간순 앞쪽 70%) → 보고서에 쓴 모델과 똑같은 모델

| 셀 | 내용 |
|---|---|
| 1 | 설정 (데이터 출처 CSV/DB, DB 저장 여부, 저장된 모델로 예측만 할지) |
| 2 | DB 접속 정보 |
| 3 | **코드 ②** DB에서 학습 데이터 꺼내기 (v_학습데이터 SELECT, 매개변수 바인딩) |
| 4 | **코드 ③** 예측 결과를 권역예측 테이블에 저장 (INSERT, 매개변수 바인딩, 트랜잭션) |
| 5~9 | 데이터 → 전처리 → 학습·모델 저장 → 예측·CSV 저장 → DB 저장 |

**결과**: `모델/logistic_v1.joblib`, `결과/권역예측_Logistic_v1.csv` (권역명, 기준일, 예측기간, 모델명, 급증확률)

## 1. 설정

In [1]:
import os
from datetime import datetime
from decimal import Decimal

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200)

# ---------- 실행 방법 ----------
SOURCE = "csv"                     # "csv" = CSV 파일에서 읽기 / "db" = DB의 v_학습데이터 뷰에서 읽기 (코드 ②)
CSV_PATH = "../학습데이터.csv"       # SOURCE = "csv"일 때 파일 경로
SAVE_TO_DB = False                 # True = 예측 결과를 DB 권역예측 테이블에 저장 (코드 ③)
USE_SAVED_MODEL = False            # True = 학습하지 않고 저장된 모델로 예측만 (새 주간 데이터가 들어왔을 때)

# ---------- 고정 설정 ----------
MODEL_NAME = "Logistic_v1"        # 권역예측 테이블의 모델명 (30자 이내)
MODEL_PATH = os.path.join("모델", "logistic_v1.joblib")
PRED_PATH = os.path.join("결과", f"권역예측_{MODEL_NAME}.csv")

FEATURES = [
    "통합", "통합_1주전", "통합_2주전", "통합_변화_1주",   # 현재·1~2주 전 출현율과 변화량
    "연중주차",                                          # 계절
    "수온_주평균", "수온_평년차", "수온_변화_1주",          # 수온·수온 평년차·수온 변화량
    "기온_주평균", "풍속_주평균", "유의파고_주평균",         # 기온·풍속·파고
]
BEST_C = {1: 0.01, 2: 0.3, 4: 0.01}   # 01_모델비교에서 valid PR-AUC로 고른 값 (예측기간: C)
TARGET = "타깃_급증_{k}주후"
THRESHOLD = 50.0                  # 대량출현 기준 (통합 출현율 %)
EXCLUDE_SPLIT = ["추가(2024)"]     # 학습에서만 제외 (예측에는 포함 → 가장 최근 주의 예측도 나옴)

# 학습 범위: "train" = 01과 같은 앞쪽 70%만으로 학습 → 보고서의 test 성능을 그대로 갖는 모델 (기본)
#            "전체"  = 정답이 있는 모든 기간(2014~2023)으로 다시 학습 (최신 자료까지 반영하고 싶을 때)
FIT_ON = "train"
TRAIN_RATIO = 0.7

# 현재 통합이 이미 50% 이상인 행은 "급증" 질문이 성립하지 않으므로 예측에서 제외 (제안서 3.6절 정의)
PREDICT_ALREADY_HIGH = False

# 데이터 오류는 DB 전처리에서 고침. 여기서는 고쳐졌는지 확인만 (포항 부이 2018-12-06~16 수온 고장)
SENSOR_CHECK = [{"관측소ID": 22106, "시작일": ["2018-11-30", "2018-12-07"]}]
TEMP_COLS = ["수온_주평균", "수온_평년차"]

### 설정 확인: 01_모델비교 결과와 고정값(BEST_C, 최종 모델)이 같은지

In [2]:
COMPARE_PATH = os.path.join("..", "결과", "모델비교_결과표.csv")
if os.path.exists(COMPARE_PATH):
    cmp = pd.read_csv(COMPARE_PATH)
    final = cmp.loc[cmp["최종"] == "★", "모델"].unique()
    chosen = (cmp[(cmp["모델"] == "Logistic") & (cmp["세트"] == "test")]
              .set_index("예측기간")["하이퍼파라미터"].map(lambda t: eval(t)["C"]).to_dict())
    ok = list(final) == ["Logistic"] and all(abs(chosen[k] - BEST_C[k]) < 1e-12 for k in BEST_C)
    print(f"01 결과: 최종 모델 {list(final)}, C {chosen}")
    print("✅ 고정값이 01 결과와 같음" if ok else "⚠️ 고정값이 01 결과와 다름 → BEST_C를 01 결과로 고칠 것")
else:
    print("01_모델비교 결과 파일이 없어 확인을 건너뜀")

01 결과: 최종 모델 ['Logistic'], C {1: 0.01, 2: 0.3, 4: 0.01}
✅ 고정값이 01 결과와 같음


## 2. DB 접속 정보 (팀 공통, 비밀번호만 바꾸거나 환경변수 MYSQL_PWD)

In [3]:
DB = dict(host="localhost", port=3306, user="root",
          password=os.environ.get("MYSQL_PWD", "비밀번호"),
          database="jellyfish", charset="utf8mb4")


def connect():
    """트랜잭션을 직접 관리하도록 autocommit을 끈 연결을 돌려준다."""
    import pymysql
    return pymysql.connect(**DB, autocommit=False)

## 3. [코드 ②] DB에서 학습 데이터 꺼내기 (v_학습데이터 뷰 SELECT, 매개변수 바인딩)

In [4]:
SQL_SELECT = """
SELECT *
FROM v_학습데이터
WHERE 시작일 >= %s
ORDER BY 권역명, 시작일
"""


def 학습데이터_조회(conn, 시작일="1900-01-01"):
    """v_학습데이터 뷰를 조회해 DataFrame으로 돌려준다. 값은 문자열로 이어 붙이지 않고 %s로 바인딩한다."""
    with conn.cursor() as cur:
        cur.execute(SQL_SELECT, (시작일,))
        cols = [c[0] for c in cur.description]
        rows = cur.fetchall()
    df = pd.DataFrame(rows, columns=cols)
    for c in df.columns:                                   # MySQL DECIMAL → 실수로 변환
        if df[c].map(lambda v: isinstance(v, Decimal)).any():
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["시작일"] = pd.to_datetime(df["시작일"])
    return df

## 4. [코드 ③] 예측 결과를 권역예측 테이블에 저장 (INSERT, 매개변수 바인딩, 트랜잭션)
- 같은 모델명의 기존 예측은 지우고 새로 넣음 → 다시 실행해도 기본키 중복 오류가 나지 않음
- 삭제와 삽입을 한 트랜잭션으로 묶어, 중간에 실패하면 전부 취소(롤백)되어 기존 예측이 그대로 남음

In [5]:
PRED_COLS = ["권역명", "기준일", "예측기간", "모델명", "급증확률"]
SQL_DELETE = "DELETE FROM 권역예측 WHERE 모델명 = %s"
SQL_INSERT = "INSERT INTO 권역예측 (권역명, 기준일, 예측기간, 모델명, 급증확률) VALUES (%s, %s, %s, %s, %s)"


def 권역예측_저장(conn, pred):
    """pred: PRED_COLS 열을 가진 DataFrame. 저장한 행 수를 돌려준다."""
    missing = [c for c in PRED_COLS if c not in pred.columns]
    if missing:
        raise ValueError(f"예측 결과에 다음 열이 없습니다: {missing}")
    if not pred["급증확률"].between(0, 1).all():
        raise ValueError("급증확률은 0~1 사이여야 합니다")
    if not pred["예측기간"].isin([1, 2, 4]).all():
        raise ValueError("예측기간은 1, 2, 4 중 하나여야 합니다")

    rows = [(str(r.권역명), str(r.기준일)[:10], int(r.예측기간), str(r.모델명), round(float(r.급증확률), 4))
            for r in pred[PRED_COLS].itertuples(index=False)]
    try:
        with conn.cursor() as cur:
            for m in sorted(pred["모델명"].unique()):
                cur.execute(SQL_DELETE, (m,))           # 같은 모델명의 이전 예측 삭제
            cur.executemany(SQL_INSERT, rows)           # 새 예측 일괄 삽입
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    return len(rows)

## 5. 데이터 읽기

In [6]:
if SOURCE == "db":
    conn = connect()
    try:
        raw = 학습데이터_조회(conn)                      # 코드 ②
    finally:
        conn.close()
else:
    try:
        raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
    except UnicodeDecodeError:
        raw = pd.read_csv(CSV_PATH, encoding="cp949")
    raw.columns = raw.columns.str.strip()
    raw["시작일"] = pd.to_datetime(raw["시작일"])
print(f"[데이터] {len(raw)}행, {raw['시작일'].min().date()} ~ {raw['시작일'].max().date()} (출처: {SOURCE})")

[데이터] 2554행, 2014-09-05 ~ 2024-05-24 (출처: csv)


## 6. 전처리: 필요한 열 확인 + 센서 고장값 처리 확인 + 수온_변화_1주 생성 (v_학습데이터의 통합_1주전과 같은 4~10일 기준)

In [7]:
def preprocess(df):
    need = ["권역명", "시작일"] + [c for c in FEATURES if c != "수온_변화_1주"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"데이터에 다음 열이 없습니다: {missing}")
    df = df.sort_values(["권역명", "시작일"]).reset_index(drop=True)
    for e in SENSOR_CHECK:                                 # DB에서 센서 고장값이 비워졌는지 확인만
        m = (df["관측소ID"] == e["관측소ID"]) & df["시작일"].isin(pd.to_datetime(e["시작일"]))
        left = df.loc[m, TEMP_COLS].notna().any(axis=1).sum()
        print(f"⚠️ 경고: 센서 고장 구간 수온 {left}행 남아 있음 → DB 전처리 확인 필요" if left
              else f"확인: 센서 고장 구간 수온 결측 처리됨 ({m.sum()}행)")
    g = df.groupby("권역명")
    gap = (df["시작일"] - g["시작일"].shift()).dt.days
    df["수온_변화_1주"] = np.where(gap.between(4, 10), df["수온_주평균"] - g["수온_주평균"].shift(), np.nan)
    return df


df = preprocess(raw)
df[["권역명", "시작일"] + FEATURES].head()

확인: 센서 고장 구간 수온 결측 처리됨 (2행)


,권역명,시작일,통합,통합_1주전,통합_2주전,통합_변화_1주,연중주차,수온_주평균,수온_평년차,수온_변화_1주,기온_주평균,풍속_주평균,유의파고_주평균
0,강원,2014-09-05,0.0,NaN,NaN,NaN,37,24.098,0.648286,NaN,22.852,2.858,0.646
1,강원,2014-09-12,0.0,0.0,NaN,0.0,38,23.324,0.032857,-0.774,22.167,4.201,0.548
2,강원,2014-09-19,0.0,0.0,0.0,0.0,39,23.485,0.420875,0.161,21.491,4.642,0.994
3,강원,2014-09-26,20.0,0.0,0.0,20.0,40,23.514,0.449875,0.029,19.991,6.160,1.459
4,강원,2014-10-03,25.0,20.0,0.0,5.0,41,NaN,NaN,NaN,NaN,NaN,NaN


## 7. 학습 → 모델 저장 (로지스틱 고정)

In [8]:
def build(C):
    return make_pipeline(
        SimpleImputer(strategy="median"),        # 결측: 학습 데이터 중앙값으로 대체
        StandardScaler(),                        # 표준화: 학습 데이터 기준
        LogisticRegression(C=C, class_weight="balanced", max_iter=2000),
    )


def train(df):
    data = df[~df["분할"].isin(EXCLUDE_SPLIT)] if "분할" in df.columns else df
    if FIT_ON == "train":   # 01과 같은 시간순 앞쪽 70%
        counts = data["시작일"].value_counts().sort_index()
        cum = data["시작일"].map(counts.cumsum()) / len(data)
        data = data[cum <= TRAIN_RATIO]

    bundle = {"모델명": MODEL_NAME, "변수": FEATURES, "C": BEST_C, "학습범위": FIT_ON,
              "학습일시": datetime.now().strftime("%Y-%m-%d %H:%M"), "모델": {}, "학습정보": {}}
    for k, C in BEST_C.items():
        y_col = TARGET.format(k=k)
        d = data.dropna(subset=[y_col])                     # 정답이 있는 행만
        model = build(C).fit(d[FEATURES], d[y_col].astype(int))
        bundle["모델"][k] = model
        bundle["학습정보"][k] = {"행수": len(d), "급증수": int(d[y_col].sum()),
                              "기간": f"{d['시작일'].min().date()} ~ {d['시작일'].max().date()}"}
        coef = pd.Series(model.named_steps["logisticregression"].coef_[0], index=FEATURES)
        bundle["학습정보"][k]["계수"] = coef.round(4).to_dict()

    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    joblib.dump(bundle, MODEL_PATH)
    print(f"[학습] {MODEL_NAME} 저장 → {MODEL_PATH}  (학습범위: {FIT_ON})")
    for k, info in bundle["학습정보"].items():
        print(f"  {k}주 뒤: C={BEST_C[k]}, {info['행수']}행 (급증 {info['급증수']}건), {info['기간']}")
    return bundle


if USE_SAVED_MODEL:
    bundle = joblib.load(MODEL_PATH)
    print(f"[불러오기] {MODEL_PATH} (학습일시 {bundle['학습일시']}, 학습범위 {bundle['학습범위']})")
else:
    bundle = train(df)

[학습] Logistic_v1 저장 → 모델/logistic_v1.joblib  (학습범위: train)
  1주 뒤: C=0.01, 1110행 (급증 89건), 2014-09-05 ~ 2021-07-23
  2주 뒤: C=0.3, 1061행 (급증 124건), 2014-09-05 ~ 2021-07-23
  4주 뒤: C=0.01, 903행 (급증 135건), 2014-09-05 ~ 2021-07-23


## 8. 예측 → CSV 저장
현재 통합이 이미 50% 이상인 권역·주는 "급증" 질문이 성립하지 않아 제외 (웹에서는 "이미 대량출현"으로 표시)

In [9]:
if bundle["변수"] != FEATURES:
    raise ValueError("저장된 모델의 입력 변수가 현재 설정과 다릅니다. 다시 학습하세요.")
target = df if PREDICT_ALREADY_HIGH else df[df["통합"] < THRESHOLD]

out = []
for k, model in bundle["모델"].items():
    out.append(pd.DataFrame({
        "권역명": target["권역명"].values,
        "기준일": target["시작일"].dt.strftime("%Y-%m-%d").values,
        "예측기간": k,
        "모델명": bundle["모델명"],
        "급증확률": model.predict_proba(target[FEATURES])[:, 1].round(4),
    }))
pred = pd.concat(out).sort_values(["예측기간", "기준일", "권역명"]).reset_index(drop=True)

os.makedirs(os.path.dirname(PRED_PATH), exist_ok=True)
pred.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")
print(f"[예측] {len(target)}개 권역·주 × {len(bundle['모델'])}기간 = {len(pred)}행 → {PRED_PATH}")
print(f"  (현재 통합 {THRESHOLD:.0f}% 이상이라 급증 예측 대상이 아닌 {len(df) - len(target)}개 권역·주는 제외)")
latest = pred[pred["기준일"] == pred["기준일"].max()]
print(f"가장 최근 기준일 {latest['기준일'].iloc[0]} 예측:")
latest.pivot(index="권역명", columns="예측기간", values="급증확률")

[예측] 1849개 권역·주 × 3기간 = 5547행 → 결과/권역예측_Logistic_v1.csv
  (현재 통합 50% 이상이라 급증 예측 대상이 아닌 705개 권역·주는 제외)
가장 최근 기준일 2024-05-24 예측:


예측기간,1,2,4
권역명,,,
경북,0.6552,0.8018,0.7421
부산,0.4232,0.6111,0.6523
울산,0.4101,0.5993,0.6445
인천,0.5033,0.7317,0.6858
제주,0.4853,0.6869,0.6943


## 9. DB 권역예측 테이블에 저장 (SAVE_TO_DB = True일 때, 코드 ③)

In [10]:
if SAVE_TO_DB:
    conn = connect()
    try:
        n = 권역예측_저장(conn, pred)                   # 코드 ③
    finally:
        conn.close()
    print(f"[저장] 권역예측 테이블에 {n}행 저장 (모델명 {MODEL_NAME})")
else:
    print("SAVE_TO_DB = False → DB 저장은 건너뜀 (CSV만 저장)")

SAVE_TO_DB = False → DB 저장은 건너뜀 (CSV만 저장)
